# レストラン来客数予測 - ランダムフォレスト提出用

## Recruit Restaurant Visitor Forecasting

**モデル**: Random Forest  
**目的**: LightGBM（04_submission.ipynb）とのスコア比較

### LightGBM版との違い
- モデルをRandomForestRegressorに変更
- NaN処理: RFはNaNを扱えないため、欠損値を補完
- log変換した目的変数で学習（RMSLEの最適化）

**データ前処理・特徴量は04_submission.ipynbと同一**

---
## 1. セットアップとデータ読み込み

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

%matplotlib inline
plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12

SEED = 42
np.random.seed(SEED)

INPUT_DIR = Path('../input')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

print('セットアップ完了')

セットアップ完了


In [2]:
# データ読み込み
air_visit = pd.read_csv(INPUT_DIR / 'air_visit_data.csv', parse_dates=['visit_date'])
air_reserve = pd.read_csv(INPUT_DIR / 'air_reserve.csv', parse_dates=['visit_datetime', 'reserve_datetime'])
air_store = pd.read_csv(INPUT_DIR / 'air_store_info.csv')
hpg_reserve = pd.read_csv(INPUT_DIR / 'hpg_reserve.csv', parse_dates=['visit_datetime', 'reserve_datetime'])
store_relation = pd.read_csv(INPUT_DIR / 'store_id_relation.csv')
date_info = pd.read_csv(INPUT_DIR / 'date_info.csv', parse_dates=['calendar_date'])
submission = pd.read_csv(INPUT_DIR / 'sample_submission.csv')

submission['air_store_id'] = submission['id'].apply(lambda x: x.rsplit('_', 1)[0])
submission['visit_date'] = pd.to_datetime(submission['id'].apply(lambda x: x.rsplit('_', 1)[1]))

print(f'来客データ: {air_visit.shape}')
print(f'提出サンプル: {submission.shape}')

来客データ: (252108, 3)
提出サンプル: (32019, 4)


---
## 2. 特徴量エンジニアリング

04_submission.ipynbと同一の特徴量を作成する。

In [3]:
# テスト・学習データの結合
train_df = air_visit[['air_store_id', 'visit_date', 'visitors']].copy()
test_df = submission[['air_store_id', 'visit_date']].copy()
test_df['visitors'] = np.nan

all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df = all_df.sort_values(['air_store_id', 'visit_date']).reset_index(drop=True)

# --- 時間特徴量 ---
all_df['year'] = all_df['visit_date'].dt.year
all_df['month'] = all_df['visit_date'].dt.month
all_df['day'] = all_df['visit_date'].dt.day
all_df['dow'] = all_df['visit_date'].dt.dayofweek
all_df['week'] = all_df['visit_date'].dt.isocalendar().week.astype(int)
all_df['is_weekend'] = (all_df['dow'] >= 5).astype(int)

all_df = all_df.merge(
    date_info.rename(columns={'calendar_date': 'visit_date'}),
    on='visit_date', how='left'
)
all_df['is_holiday'] = all_df['holiday_flg'].fillna(0).astype(int)

date_info_sorted = date_info.sort_values('calendar_date')
date_info_sorted['is_before_holiday'] = date_info_sorted['holiday_flg'].shift(-1).fillna(0).astype(int)
all_df = all_df.merge(
    date_info_sorted[['calendar_date', 'is_before_holiday']].rename(columns={'calendar_date': 'visit_date'}),
    on='visit_date', how='left'
)
all_df['is_before_holiday'] = all_df['is_before_holiday'].fillna(0).astype(int)

# --- 店舗情報 ---
all_df = all_df.merge(air_store, on='air_store_id', how='left')
all_df['genre_encoded'] = all_df['air_genre_name'].factorize()[0]
all_df['area_encoded'] = all_df['air_area_name'].factorize()[0]
all_df['prefecture'] = all_df['air_area_name'].apply(lambda x: x.split(' ')[0] if pd.notna(x) else 'unknown')
all_df['prefecture_encoded'] = all_df['prefecture'].factorize()[0]

# --- 店舗統計量 ---
store_stats = train_df.groupby('air_store_id')['visitors'].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
).reset_index()
store_stats.columns = ['air_store_id', 'store_mean', 'store_median', 'store_std',
                        'store_min', 'store_max', 'store_count']
all_df = all_df.merge(store_stats, on='air_store_id', how='left')

train_with_dow = train_df.copy()
train_with_dow['dow'] = train_with_dow['visit_date'].dt.dayofweek
store_dow_stats = train_with_dow.groupby(['air_store_id', 'dow'])['visitors'].agg(
    ['mean', 'median', 'std']
).reset_index()
store_dow_stats.columns = ['air_store_id', 'dow', 'store_dow_mean', 'store_dow_median', 'store_dow_std']
all_df = all_df.merge(store_dow_stats, on=['air_store_id', 'dow'], how='left')

genre_stats = train_df.merge(air_store[['air_store_id', 'air_genre_name']], on='air_store_id')
genre_stats = genre_stats.groupby('air_genre_name')['visitors'].agg(['mean', 'median']).reset_index()
genre_stats.columns = ['air_genre_name', 'genre_mean', 'genre_median']
all_df = all_df.merge(genre_stats, on='air_genre_name', how='left')

print(f'店舗統計まで: {all_df.shape}')

店舗統計まで: (284127, 32)


In [4]:
# --- 予約情報 ---
air_reserve['visit_date'] = pd.to_datetime(air_reserve['visit_datetime'].dt.date)
air_reserve['reserve_date'] = pd.to_datetime(air_reserve['reserve_datetime'].dt.date)
air_reserve['days_before'] = (air_reserve['visit_date'] - air_reserve['reserve_date']).dt.days

air_res_agg = air_reserve.groupby(['air_store_id', 'visit_date']).agg(
    air_reserve_visitors=('reserve_visitors', 'sum'),
    air_reserve_count=('reserve_visitors', 'count'),
).reset_index()

hpg_reserve['visit_date'] = pd.to_datetime(hpg_reserve['visit_datetime'].dt.date)
hpg_reserve['reserve_date'] = pd.to_datetime(hpg_reserve['reserve_datetime'].dt.date)
hpg_reserve = hpg_reserve.merge(store_relation, on='hpg_store_id', how='inner')

hpg_res_agg = hpg_reserve.groupby(['air_store_id', 'visit_date']).agg(
    hpg_reserve_visitors=('reserve_visitors', 'sum'),
    hpg_reserve_count=('reserve_visitors', 'count'),
).reset_index()

all_df = all_df.merge(air_res_agg, on=['air_store_id', 'visit_date'], how='left')
all_df = all_df.merge(hpg_res_agg, on=['air_store_id', 'visit_date'], how='left')
all_df['total_reserve_visitors'] = all_df['air_reserve_visitors'].fillna(0) + all_df['hpg_reserve_visitors'].fillna(0)
all_df['total_reserve_count'] = all_df['air_reserve_count'].fillna(0) + all_df['hpg_reserve_count'].fillna(0)

print(f'予約情報まで: {all_df.shape}')

予約情報まで: (284127, 38)


In [5]:
# --- Rolling統計量とラグ特徴量 ---
all_stores = all_df['air_store_id'].unique()
min_date = all_df['visit_date'].min()
max_date = all_df['visit_date'].max()
all_dates = pd.date_range(min_date, max_date, freq='D')

grid = pd.MultiIndex.from_product([all_stores, all_dates], names=['air_store_id', 'visit_date'])
grid_df = pd.DataFrame(index=grid).reset_index()
grid_df = grid_df.merge(train_df[['air_store_id', 'visit_date', 'visitors']],
                         on=['air_store_id', 'visit_date'], how='left')
grid_df = grid_df.sort_values(['air_store_id', 'visit_date']).reset_index(drop=True)

for w in [7, 14, 21, 35, 63]:
    grouped = grid_df.groupby('air_store_id')['visitors']
    grid_df[f'rolling_mean_{w}'] = grouped.transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    grid_df[f'rolling_std_{w}'] = grouped.transform(lambda x: x.shift(1).rolling(w, min_periods=1).std())
    grid_df[f'rolling_max_{w}'] = grouped.transform(lambda x: x.shift(1).rolling(w, min_periods=1).max())
    grid_df[f'rolling_min_{w}'] = grouped.transform(lambda x: x.shift(1).rolling(w, min_periods=1).min())

grid_df['ewm_mean'] = grid_df.groupby('air_store_id')['visitors'].transform(
    lambda x: x.shift(1).ewm(span=14, min_periods=1).mean()
)

for lag in [1, 7, 14, 21, 28, 35]:
    grid_df[f'lag_{lag}'] = grid_df.groupby('air_store_id')['visitors'].shift(lag)

rolling_cols = [c for c in grid_df.columns if c.startswith(('rolling_', 'ewm_', 'lag_'))]
all_df = all_df.merge(grid_df[['air_store_id', 'visit_date'] + rolling_cols],
                       on=['air_store_id', 'visit_date'], how='left')

print(f'最終データ: {all_df.shape}')

最終データ: (284127, 65)


### 検証: 休店日の取り扱い確認

休店日（air_visit_dataに存在しない日付）が rolling / lag 特徴量でどう扱われているかを実データで確認する。

**確認ポイント:**
1. grid_df で休店日の visitors が NaN になっているか
2. rolling_mean がNaNをスキップして計算されているか
3. lag_1 が休店日翌日で NaN になるか
4. all_df に休店日の行が含まれないか

In [6]:
# === 休店日の取り扱い検証 ===

# 1. 特定の店舗を選び、休店日を特定
store_id = air_visit['air_store_id'].iloc[0]
store_dates = set(air_visit[air_visit['air_store_id'] == store_id]['visit_date'])
all_dates_set = set(pd.date_range(air_visit['visit_date'].min(), air_visit['visit_date'].max()))
closed_days = sorted(all_dates_set - store_dates)

print(f'店舗: {store_id}')
print(f'営業日数: {len(store_dates)}, 全日数: {len(all_dates_set)}, 休店日数: {len(closed_days)}')
print(f'休店率: {len(closed_days)/len(all_dates_set)*100:.1f}%')
print(f'\n最初の5休店日: {[d.strftime("%Y-%m-%d (%a)") for d in closed_days[:5]]}')

# 2. 休店日の前後でgrid_dfを確認（最初の休店日を使用）
closed_day = closed_days[0]
window = pd.Timedelta(days=3)
mask = (
    (grid_df['air_store_id'] == store_id)
    & (grid_df['visit_date'].between(closed_day - window, closed_day + window))
)
display_cols = ['visit_date', 'visitors', 'rolling_mean_7', 'rolling_std_7', 'lag_1', 'lag_7']
print(f'\n--- grid_df: 休店日 {closed_day.strftime("%Y-%m-%d (%a)")} の前後 ---')
display(grid_df.loc[mask, display_cols].to_string(index=False))

# 3. all_dfに同じ休店日の行が含まれないことを確認
closed_in_all = all_df[
    (all_df['air_store_id'] == store_id) & (all_df['visit_date'] == closed_day)
]
print(f'\n--- all_df に休店日 {closed_day.strftime("%Y-%m-%d")} の行があるか: {len(closed_in_all) > 0} (行数: {len(closed_in_all)}) ---')

# 4. 休店日翌日のall_dfの特徴量を確認（lag_1がNaNになるか）
next_day = closed_day + pd.Timedelta(days=1)
next_in_all = all_df[
    (all_df['air_store_id'] == store_id) & (all_df['visit_date'] == next_day)
]
if len(next_in_all) > 0:
    print(f'\n--- all_df: 休店日翌日 {next_day.strftime("%Y-%m-%d (%a)")} の特徴量 ---')
    lag_cols = ['visit_date', 'visitors', 'lag_1', 'lag_7', 'rolling_mean_7']
    display(next_in_all[lag_cols].to_string(index=False))
    print(f'  lag_1 = {next_in_all["lag_1"].values[0]} (前日が休店なのでNaNのはず)')
else:
    print(f'\n休店日翌日 {next_day.strftime("%Y-%m-%d")} も休店（all_dfに行なし）')
    # 連続休店の場合、次の営業日を探す
    for offset in range(2, 8):
        check_day = closed_day + pd.Timedelta(days=offset)
        check_in_all = all_df[
            (all_df['air_store_id'] == store_id) & (all_df['visit_date'] == check_day)
        ]
        if len(check_in_all) > 0:
            print(f'  次の営業日: {check_day.strftime("%Y-%m-%d (%a)")}')
            print(f'  lag_1 = {check_in_all["lag_1"].values[0]}')
            print(f'  rolling_mean_7 = {check_in_all["rolling_mean_7"].values[0]}')
            break

# 5. サマリー統計
print('\n=== サマリー ===')
print(f'grid_dfの行数: {len(grid_df)} (全店舗×全日付)')
print(f'all_dfの行数:  {len(all_df)} (営業日のみ + テスト期間)')
print(f'grid_dfでvisitors=NaNの行数: {grid_df["visitors"].isna().sum()} (休店日 + テスト期間)')
print(f'all_dfでlag_1=NaNの行数:     {all_df["lag_1"].isna().sum()}')

店舗: air_ba937bf13d40fb24
営業日数: 391, 全日数: 478, 休店日数: 87
休店率: 18.2%

最初の5休店日: ['2016-01-01 (Fri)', '2016-01-02 (Sat)', '2016-01-03 (Sun)', '2016-01-04 (Mon)', '2016-01-05 (Tue)']

--- grid_df: 休店日 2016-01-01 (Fri) の前後 ---


'visit_date  visitors  rolling_mean_7  rolling_std_7  lag_1  lag_7\n2016-01-01       NaN             NaN            NaN    NaN    NaN\n2016-01-02       NaN             NaN            NaN    NaN    NaN\n2016-01-03       NaN             NaN            NaN    NaN    NaN\n2016-01-04       NaN             NaN            NaN    NaN    NaN'


--- all_df に休店日 2016-01-01 の行があるか: False (行数: 0) ---

休店日翌日 2016-01-02 も休店（all_dfに行なし）

=== サマリー ===
grid_dfの行数: 428593 (全店舗×全日付)
all_dfの行数:  284127 (営業日のみ + テスト期間)
grid_dfでvisitors=NaNの行数: 176485 (休店日 + テスト期間)
all_dfでlag_1=NaNの行数:     59765


---
## 3. ランダムフォレストによるモデル学習

### ランダムフォレストの特徴
- **バギング**: 複数の決定木を独立に学習し、予測を平均化
- **利点**: 外れ値に強い、過学習しにくい、スケーリング不要
- **欠点**: 勾配ブースティングより精度が劣る傾向、NaN不可

### NaN処理
scikit-learnのRandomForestRegressorはNaNを受け付けないため、欠損値を補完する必要がある。

In [7]:
FEATURE_COLS = [
    'month', 'day', 'dow', 'week', 'is_weekend', 'is_holiday', 'is_before_holiday',
    'genre_encoded', 'area_encoded', 'prefecture_encoded',
    'latitude', 'longitude',
    'store_mean', 'store_median', 'store_std', 'store_min', 'store_max', 'store_count',
    'store_dow_mean', 'store_dow_median', 'store_dow_std',
    'genre_mean', 'genre_median',
    'air_reserve_visitors', 'air_reserve_count',
    'hpg_reserve_visitors', 'hpg_reserve_count',
    'total_reserve_visitors', 'total_reserve_count',
    'rolling_mean_7', 'rolling_std_7', 'rolling_max_7', 'rolling_min_7',
    'rolling_mean_14', 'rolling_std_14', 'rolling_max_14', 'rolling_min_14',
    'rolling_mean_21', 'rolling_std_21', 'rolling_max_21', 'rolling_min_21',
    'rolling_mean_35', 'rolling_std_35', 'rolling_max_35', 'rolling_min_35',
    'rolling_mean_63', 'rolling_std_63', 'rolling_max_63', 'rolling_min_63',
    'ewm_mean',
    'lag_1', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_35',
]

TARGET = 'visitors'

is_train = all_df[TARGET].notna()
train = all_df[is_train].copy()
test = all_df[~is_train].copy()

# NaN補完（RFはNaN不可）: -999で補完
train[FEATURE_COLS] = train[FEATURE_COLS].fillna(-999)
test[FEATURE_COLS] = test[FEATURE_COLS].fillna(-999)

print(f'学習データ: {train.shape}')
print(f'テストデータ: {test.shape}')
print(f'NaN残存: {train[FEATURE_COLS].isna().sum().sum()}')

学習データ: (252108, 65)
テストデータ: (32019, 65)
NaN残存: 0


### 3.1 バリデーション

In [8]:
VALID_START = '2017-03-12'

tr_mask = train['visit_date'] < VALID_START
va_mask = train['visit_date'] >= VALID_START

X_tr = train.loc[tr_mask, FEATURE_COLS]
y_tr = train.loc[tr_mask, TARGET]
X_va = train.loc[va_mask, FEATURE_COLS]
y_va = train.loc[va_mask, TARGET]

print(f'学習: {X_tr.shape}')
print(f'検証: {X_va.shape}')

学習: (222073, 56)
検証: (30035, 56)


### 3.1b グリッドサーチによるパラメータ最適化

時系列分割（train < 2017-03-12, valid >= 2017-03-12）を使い、GridSearchCVでハイパーパラメータを探索する。

- `PredefinedSplit` で単一の時系列分割を実現（ランダム分割ではない）
- スコアリング: `neg_mean_squared_error`（log空間のMSE = RMSLE²）
- `refit=False`: 最適パラメータの特定のみ。再学習は後続セルで行う

In [9]:
%%time
from sklearn.model_selection import GridSearchCV, PredefinedSplit

# PredefinedSplit: -1 = 常に学習, 0 = 検証
test_fold = np.where(train['visit_date'] < VALID_START, -1, 0)
ps = PredefinedSplit(test_fold)

param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [3, 5, 10],
    'max_features': ['sqrt', 'log2', 0.5],
}

total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)
print(f'探索パラメータ組み合わせ数: {total_combos}')
print(f'各組み合わせ1回のみフィット（PredefinedSplit）\n')

rf_base = RandomForestRegressor(n_jobs=-1, random_state=SEED)

gs = GridSearchCV(
    rf_base,
    param_grid,
    cv=ps,
    scoring='neg_mean_squared_error',
    refit=False,
    verbose=1,
    n_jobs=1,  # RF側でn_jobs=-1を使うため
)

gs.fit(train[FEATURE_COLS], np.log1p(train[TARGET]))

best_params = gs.best_params_
best_rmsle = np.sqrt(-gs.best_score_)

print(f'\n========================================')
print(f'GridSearchCV 結果')
print(f'  ベスト検証 RMSLE: {best_rmsle:.5f}')
print(f'  ベストパラメータ:')
for k, v in best_params.items():
    print(f'    {k}: {v}')
print(f'========================================')

# 上位10組み合わせを表示
results_df = pd.DataFrame(gs.cv_results_)
results_df['rmsle'] = np.sqrt(-results_df['mean_test_score'])
results_df = results_df.sort_values('rmsle')
print(f'\n上位10パラメータ組み合わせ:')
for i, (_, row) in enumerate(results_df.head(10).iterrows()):
    print(f'  {i+1}. RMSLE={row["rmsle"]:.5f}  {row["params"]}')

探索パラメータ組み合わせ数: 324
各組み合わせ1回のみフィット（PredefinedSplit）

Fitting 1 folds for each of 324 candidates, totalling 324 fits
CPU times: total: 14h 25min 21s
Wall time: 6h 14min 45s


KeyboardInterrupt: 

In [ ]:
%%time

# ランダムフォレスト（GridSearchCVのベストパラメータで学習）
rf_model = RandomForestRegressor(
    **best_params,
    n_jobs=-1,
    random_state=SEED
)

rf_model.fit(X_tr, np.log1p(y_tr))

# 検証
va_pred = np.expm1(rf_model.predict(X_va))
va_pred = np.clip(va_pred, 0, None)
va_score = rmsle(y_va, va_pred)

tr_pred = np.expm1(rf_model.predict(X_tr))
tr_score = rmsle(y_tr, tr_pred)

print(f'\n=====================================')
print(f'Random Forest 結果 (最適パラメータ)')
print(f'  パラメータ: {best_params}')
print(f'  学習 RMSLE: {tr_score:.5f}')
print(f'  検証 RMSLE: {va_score:.5f}')
print(f'  過学習度:   {va_score - tr_score:.5f}')
print(f'=====================================')

### 3.2 特徴量重要度

In [ ]:
importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(importance['feature'], importance['importance'], color='#55A868')
ax.set_xlabel('重要度 (Impurity Decrease)')
ax.set_title('Random Forest 特徴量重要度')
plt.tight_layout()
plt.show()

print('\n上位10特徴量:')
for _, row in importance.tail(10).iloc[::-1].iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')

### 3.3 全データで再学習（提出用）

In [ ]:
%%time

# 全学習データで最終モデル（GridSearchCVのベストパラメータ）
X_all = train[FEATURE_COLS]
y_all = train[TARGET]

final_rf = RandomForestRegressor(
    **best_params,
    n_jobs=-1,
    random_state=SEED
)

final_rf.fit(X_all, np.log1p(y_all))
print(f'最終モデル学習完了')
print(f'パラメータ: {best_params}')

---
## 4. 予測と提出ファイル生成

In [ ]:
# テストデータの予測
X_test = test[FEATURE_COLS]
test_pred = np.expm1(final_rf.predict(X_test))
test_pred = np.clip(test_pred, 0, None)

print(f'予測統計:')
print(f'  平均: {test_pred.mean():.1f}')
print(f'  中央値: {np.median(test_pred):.1f}')
print(f'  最小: {test_pred.min():.1f}')
print(f'  最大: {test_pred.max():.1f}')

In [ ]:
# 提出ファイル作成
sub = submission[['id']].copy()

test_with_id = test.copy()
test_with_id['id'] = test_with_id['air_store_id'] + '_' + test_with_id['visit_date'].dt.strftime('%Y-%m-%d')
test_with_id['visitors'] = test_pred

sub = sub.merge(test_with_id[['id', 'visitors']], on='id', how='left')

missing = sub['visitors'].isna().sum()
if missing > 0:
    print(f'警告: {missing}件の欠損あり。店舗平均で補完します。')
    sub['air_store_id'] = sub['id'].apply(lambda x: x.rsplit('_', 1)[0])
    sub = sub.merge(store_stats[['air_store_id', 'store_mean']], on='air_store_id', how='left')
    sub['visitors'] = sub['visitors'].fillna(sub['store_mean']).fillna(train[TARGET].mean())
    sub = sub[['id', 'visitors']]

output_path = OUTPUT_DIR / 'submission_rf.csv'
sub.to_csv(output_path, index=False)

print(f'\n提出ファイルを保存: {output_path}')
print(f'レコード数: {len(sub)}')
print(f'\n先頭5行:')
sub.head()

In [ ]:
# 分布比較
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(train[TARGET], bins=100, alpha=0.5, label='学習データ（実績）', color='#4C72B0')
ax.hist(sub['visitors'], bins=100, alpha=0.5, label='RF予測', color='#55A868')
ax.set_title('来客数の分布比較')
ax.set_xlabel('来客数')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. まとめ

### Random Forest の結果
- 検証RMSLE: 上記セルの出力を参照
- 提出ファイル: `submission_rf.csv`

### LightGBM (04_submission.ipynb) との比較
| モデル | 検証RMSLE | 提出ファイル |
|--------|-----------|-------------|
| LightGBM | 0.47392 | submission.csv |
| Random Forest | 上記参照 | submission_rf.csv |

### RF vs LightGBM の差が生まれる理由
1. **バギング vs ブースティング**: RFは独立した木の平均、LightGBMは誤差を逐次修正
2. **NaN処理**: LightGBMはNaNを自然に扱えるが、RFは補完が必要
3. **計算効率**: RFは全データ点を使うが、LightGBMはヒストグラム化で高速化

In [ ]:
print('=== 完了 ===')
print(f'提出ファイル: {output_path}')
print(f'\nKaggle提出コマンド:')
print(f'kaggle competitions submit -c recruit-restaurant-visitor-forecasting -f {output_path} -m "Random Forest baseline"')